# 27 · A2A: agent discovery and agent cards
Level L5 · Part 5 · professional · ~25 min · builds on 21

## Goal
Publish an agent card for each of two Brightside agents on localhost, let a caller discover both at the well-known URL,
and read each agent's skills and sign-in requirements before it sends a task. No task is sent in this lesson; lesson 27a does that.

## Why it matters
Lesson 21 gave one app one set of tools over MCP. Now the refund desk is a separate agent, owned by another team and
running on another server. The support agent has to answer three questions before it can ask for a refund: where is the
refund agent, what can it do, and does it need a token. Without a standard, the caller hard-codes all three answers:

In [1]:
# Without discovery: the caller hard-codes the URL, the skill name and the sign-in rule for every remote agent.
HARD_CODED = {"refunds": {"url": "http://refunds.internal:9000/task", "skill": "issue_refund", "token": True}}
print("the answers the caller guessed:", HARD_CODED["refunds"])
print("the team renames the skill to 'refund_order'. The caller still sends 'issue_refund' and nothing tells it.")

the answers the caller guessed: {'url': 'http://refunds.internal:9000/task', 'skill': 'issue_refund', 'token': True}
the team renames the skill to 'refund_order'. The caller still sends 'issue_refund' and nothing tells it.


| Without an agent card | With an agent card |
|---|---|
| URL, skill names and sign-in rules copied into the caller | The caller fetches one JSON document from the agent |
| A rename on the other team breaks the caller at run time | The caller reads the current skill ids on every start |
| A new agent means a code change in every caller | A new agent is one more URL in a list |

**What it makes easy:** a caller can find an agent it has never seen and check what it needs before it sends anything.
**When you don't need it:** one agent, one team, one repository. Plain function calls (or the MCP tools of lesson 21) are enough.

## The idea
An **A2A** agent (Agent2Agent, the open protocol for agents talking to other agents) publishes an **agent card**: a JSON
document that says who the agent is, where its endpoint is, which **skills** it offers, what it supports (streaming,
push notifications), which **security schemes** it accepts and which skills need them. The card lives at the
well-known URL `/.well-known/agent-card.json`, so a caller only needs the agent's base URL to find it. **Discovery** is
the step in between: fetch the card, read it, then decide whether and how to call the agent. A **registry** is a list of
card URLs that a caller keeps (a config file, a directory service, a company catalogue). The card is the standard part;
finding cards is up to you.

Analogy: a business card with a list of services and a note saying "bring an ID for the refund service". You read it
before you walk into the office.

## Picture
```mermaid
flowchart LR
  C[Support agent: the caller] -->|1. GET base URL + /.well-known/agent-card.json| O[Orders agent :8270]
  C -->|1. same request| R[Refunds agent :8271]
  O -->|2. card: skill lookup_order, no sign-in| C
  R -->|2. card: skill issue_refund, needs bearer token| C
  C -->|3. decide: send a task now, or sign in first| D[Next lesson 27a: send a task]
```

## Step by step
**Step 1 · the two agents.** The orders agent is public: one skill, `lookup_order`, no sign-in. The refunds agent asks
for a bearer token on its `issue_refund` skill only. The skill ids are the same names lesson 21 gave the MCP tools, so
the story stays the same. The card is built by one function in `apps/27/support_agents.py`, which the next cell writes.
Read it once: it is the whole agent card. It serves the card with `create_agent_card_routes` from the `a2a` SDK. The
SDK's `AgentCard` is a protobuf message, so the fields are typed and checked.

In [2]:
from pathlib import Path

Path("../apps/27").mkdir(parents=True, exist_ok=True)  # the server file goes here
Path("../data/27").mkdir(parents=True, exist_ok=True)  # run-time logs go here

**The package.** This lesson uses the `a2a-sdk` package, version 1.2.3 or newer (`pip install "a2a-sdk>=1.2.3"`). The
1.x SDK speaks A2A 1.0. Older examples use `url=` on the card and other route helpers; they will not run on 1.x (see
Common mistakes).

In [3]:
%%writefile ../apps/27/support_agents.py
"""Two Brightside support agents, each publishing an A2A agent card (lesson 27).
Run: python support_agents.py <orders|refunds> <port>. Only the card is served here; tasks come in lesson 27a."""
import sys

import uvicorn
from a2a.server.routes import create_agent_card_routes
from a2a.types import (AgentCapabilities, AgentCard, AgentInterface, AgentSkill, HTTPAuthSecurityScheme,
                       SecurityRequirement, SecurityScheme, StringList)
from a2a.utils.constants import PROTOCOL_VERSION_CURRENT, TransportProtocol
from starlette.applications import Starlette

AGENTS = {"orders": "order lookup", "refunds": "refund requests"}


def build_card(which: str, port: int) -> AgentCard:
    """The card a caller reads first: who the agent is, where its endpoint is, what it can do, how to sign in."""
    refund_skill = AgentSkill(
        id="issue_refund", name="Issue refund", tags=["refund", "write"],
        description="Start a refund for one order. Needs a bearer token with the refunds scope.",
        examples=["Refund order 1002, it arrived broken"],
        security_requirements=[SecurityRequirement(schemes={"bearer": StringList(list=["refunds"])})],
    )
    lookup_skill = AgentSkill(
        id="lookup_order", name="Look up order", tags=["order", "read"],
        description="Status, days late and total for one order. Public, no sign-in.",
        examples=["Where is order 1003?"],
    )
    skills = [lookup_skill] if which == "orders" else [refund_skill]
    return AgentCard(
        name=f"Brightside {which} agent", description=f"Brightside Outfitters: {AGENTS[which]}.", version="1.0.0",
        supported_interfaces=[AgentInterface(url=f"http://127.0.0.1:{port}/", protocol_binding=TransportProtocol.JSONRPC,
                                             protocol_version=PROTOCOL_VERSION_CURRENT)],
        capabilities=AgentCapabilities(streaming=False, push_notifications=False),
        default_input_modes=["text/plain"], default_output_modes=["text/plain"],
        skills=skills,
        security_schemes={"bearer": SecurityScheme(http_auth_security_scheme=HTTPAuthSecurityScheme(
            scheme="Bearer", description="Token from the Brightside sign-in server."))},
    )


if __name__ == "__main__":
    which, port = sys.argv[1], int(sys.argv[2])
    app = Starlette(routes=create_agent_card_routes(build_card(which, port)))  # serves /.well-known/agent-card.json
    uvicorn.run(app, host="127.0.0.1", port=port, log_level="warning")

Overwriting ../apps/27/support_agents.py


**Step 2 · start both agents.** Each agent runs in its own process on its own port: 8270 for orders, 8271 for refunds
(this lesson's ports, `progress.py --ports 27`). The notebook checks each port is free, starts the agent, and polls its
card URL until it answers with HTTP 200. It does not sleep for a fixed time. The logs go to `../data/27/`.

In [4]:
import socket
import subprocess
import sys
import time

import httpx

SERVER = Path("../apps/27/support_agents.py").resolve()
ORDERS_URL = "http://127.0.0.1:8270"  # the base URL; the card is at ORDERS_URL + /.well-known/agent-card.json
REFUNDS_URL = "http://127.0.0.1:8271"


def port_open(port: int) -> bool:
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

The helper starts one agent and waits until its card answers. A dead process stops the wait early, so a bad start
shows up in the log file, not as a silent hang.

In [5]:
def start_agent(which: str, base_url: str) -> subprocess.Popen:
    port = int(base_url.rsplit(":", 1)[1])
    if port_open(port):
        raise RuntimeError(f"port {port} is busy: stop the old process first")
    log = open(f"../data/27/{which}.log", "w")  # server errors go here, not into the notebook
    proc = subprocess.Popen([sys.executable, str(SERVER), which, str(port)], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(100):  # poll for up to 10 seconds
        try:
            if httpx.get(f"{base_url}/.well-known/agent-card.json", timeout=0.5).status_code == 200:
                break
        except httpx.HTTPError:
            pass
        if proc.poll() is not None:  # the process died: stop waiting
            break
        time.sleep(0.1)
    return proc

Start both agents, then check that each card answers with HTTP 200.

In [6]:
agents = [start_agent("orders", ORDERS_URL), start_agent("refunds", REFUNDS_URL)]
for url in (ORDERS_URL, REFUNDS_URL):
    print(url, "card answers:", httpx.get(f"{url}/.well-known/agent-card.json").status_code)

http://127.0.0.1:8270 card answers: 200
http://127.0.0.1:8271 card answers: 200


**Step 3 · discover one agent.** The caller uses `A2ACardResolver` from the SDK. Give it the base URL; it adds the
well-known path, fetches the card and parses it. Below, the raw JSON of the refunds card, a peek of the first 400
characters, then the fields the caller needs: each interface's URL, each skill's id and whether it needs a token.

In [7]:
import asyncio
import json

from a2a.client import A2ACardResolver
from google.protobuf.json_format import MessageToDict


async def fetch_card(base_url: str):
    async with httpx.AsyncClient(timeout=5) as http:  # one HTTP client for the caller
        return await A2ACardResolver(http, base_url).get_agent_card()


refunds_card = await fetch_card(REFUNDS_URL)  # top-level await works in a notebook
print(json.dumps(MessageToDict(refunds_card), indent=1)[:400], "…")
print("\nname:", refunds_card.name)
print("endpoint:", refunds_card.supported_interfaces[0].url, "|", refunds_card.supported_interfaces[0].protocol_binding)
for skill in refunds_card.skills:
    print("skill:", skill.id, "| needs a token:", bool(skill.security_requirements))

{
 "name": "Brightside refunds agent",
 "description": "Brightside Outfitters: refund requests.",
 "supportedInterfaces": [
  {
   "url": "http://127.0.0.1:8271/",
   "protocolBinding": "JSONRPC",
   "protocolVersion": "1.0"
  }
 ],
 "version": "1.0.0",
 "capabilities": {
  "streaming": false,
  "pushNotifications": false
 },
 "securitySchemes": {
  "bearer": {
   "httpAuthSecurityScheme": {
    " …

name: Brightside refunds agent
endpoint: http://127.0.0.1:8271/ | JSONRPC
skill: issue_refund | needs a token: True


Read the output. The card names the endpoint (`supported_interfaces`), the skill ids a caller must use, and the sign-in
rule on the skill. Nothing about the refunds agent was copied into the caller. The card's endpoint is where lesson 27a
will send tasks; the server in this lesson does not serve that endpoint yet, so do not send anything to it here.

**Step 4 · a registry of cards.** A registry here is a list of base URLs. The caller reads every card, and builds an
index from skill id to the agent that offers it. The third URL has no server behind it, so it stands for a stale entry. The
loop skips it and keeps going: one dead agent must not stop the others from being found.

In [8]:
from a2a.client.errors import AgentCardResolutionError

REGISTRY = [ORDERS_URL, REFUNDS_URL, "http://127.0.0.1:8272"]  # 8272: a stale entry, nothing listens there


async def build_index(urls: list[str]) -> dict:
    index = {}
    async with httpx.AsyncClient(timeout=3) as http:
        for url in urls:
            try:
                card = await A2ACardResolver(http, url).get_agent_card()
            except AgentCardResolutionError as err:  # not reachable, or not a valid card
                print("skipped", url, "|", str(err)[:60], "…")
                continue
            for skill in card.skills:
                index[skill.id] = {"agent": card.name, "url": url, "needs_token": bool(skill.security_requirements)}
    return index


INDEX = await build_index(REGISTRY)
for skill_id, entry in INDEX.items():
    print(f"{skill_id:14} -> {entry['agent']} at {entry['url']} | token first: {entry['needs_token']}")

skipped http://127.0.0.1:8272 | Network communication error fetching agent card from http:// …
lookup_order   -> Brightside orders agent at http://127.0.0.1:8270 | token first: False
issue_refund   -> Brightside refunds agent at http://127.0.0.1:8271 | token first: True


**Step 5 · decide before you send.** The support agent wants a refund. It looks up `issue_refund` in the index and
reads the answer. A token is needed, so the next step would be to sign in (lesson 26 covers the sign-in and lesson 27a
the task). The caller never sends the request before it has read the card. The sign-in is not run here: there is no
token in this lesson, and none is needed to discover an agent.

In [9]:
entry = INDEX["issue_refund"]
if entry["needs_token"]:
    print("issue_refund needs a token from the sign-in server: get it first, then send the task (lessons 26 and 27a).")
else:
    print("issue_refund is open: send the task to", entry["url"])

issue_refund needs a token from the sign-in server: get it first, then send the task (lessons 26 and 27a).


## At work
- Google's Agent Development Kit (ADK 2.x) is A2A-native: it can expose an agent with a card and call agents that publish
  one. The card is how the two sides find each other without a shared codebase. → lesson 27a for the task that follows the card.
- Per-skill sign-in rules (`securityRequirements` on a skill) keep a public skill public and a refund skill behind a token.
  Production teams add signed cards and per-agent identity on top. → lessons 26 (sign-in) and 31 (agent identity, audit log).
- A2A joined the Linux Foundation's Agentic AI Foundation on 2026-08-27, where MCP already sits. Check which A2A version a
  partner speaks before you pick a client. → lesson 21 for the MCP side.
- Go deeper: signed agent cards (JWS), extended cards for signed-in callers, and registries that check cards before listing.

## What just happened
Two agents published a card at the well-known URL. A caller found them from their base URLs, read their skills and
sign-in rules, and skipped an agent that was not running. The caller did not know any skill name in advance; it read them.
The card is the contract between the two teams. The next lesson uses the same card to send a task.

**Interview:** "How does agent A find out what agent B can do, and what it needs, before it calls it?" Strong answer:
"B publishes an agent card at `/.well-known/agent-card.json` with skills, the endpoint, and the sign-in scheme per skill.
A fetches it, checks the skill and the token rule, then sends the task."

## Common mistakes
**Mistake 1 · the card field `url`.** Older A2A code put the endpoint in a `url` field on the card. In the 1.x SDK the
endpoint is in `supported_interfaces`, and the old field is refused. The cell below fails on purpose.

In [10]:
from a2a.types import AgentCard

AgentCard(name="old card", url="http://127.0.0.1:8270")  # the pre-1.0 field: fails on the 1.x SDK

ValueError: Protocol message AgentCard has no "url" field.

The fix: put the URL in `AgentInterface(url=..., protocol_binding=..., protocol_version=...)`, as `build_card` does.

**Mistake 2 · the old card path.** Earlier drafts of A2A served the card at `/.well-known/agent.json`. The 1.x agent
does not serve that path: asking for it returns 404, which the resolver turns into an error. The cell below fails on purpose.

In [11]:
await A2ACardResolver(httpx.AsyncClient(timeout=3), REFUNDS_URL, agent_card_path="/.well-known/agent.json").get_agent_card()

AgentCardResolutionError: Failed to fetch agent card from http://127.0.0.1:8271/.well-known/agent.json (HTTP 404): Client error '404 Not Found' for url 'http://127.0.0.1:8271/.well-known/agent.json'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

The fix: use `AGENT_CARD_WELL_KNOWN_PATH` from `a2a.utils.constants` (the default of `A2ACardResolver`), which is
`/.well-known/agent-card.json`. Do not hard-code the path in the caller.

**Mistake 3 · trusting the card text.** The description of a skill is text written by another team. If a model reads
it, a hostile card can carry instructions (lesson 30 covers this). Treat card text as data, and check the sign-in rule in
the code, not in the prose.

## Production checklist
- Serve the card over HTTPS at a stable host name. The card's endpoint is the address callers will use, so get it right
  before you publish.
- Put sign-in requirements on each skill (`security_requirements`), not only at the agent level, so a public skill stays public.
- Cache the card with the `ETag` header that the SDK sends, and re-fetch when a call fails. Set `Cache-Control` to a
  value that matches how often you change the card.
- Keep the registry in one reviewed place: each entry has an owner, and a dead entry gets removed, not ignored.
- Sign the card when callers cannot trust the network path (A2A supports card signatures; the `a2a` SDK can verify them).
- Never put secrets, internal host names or customer data in a card. Anyone who can fetch it can read it.

## Try it
Write `find_agent(index, skill_id)`. It returns a short sentence: which agent offers the skill, its URL, and whether a token
is needed, or "no agent offers <skill>" when the skill is missing. Test it on `lookup_order`, `issue_refund` and `track_parcel`.

In [12]:
# your code here

**Answer** — try it first, then expand the hidden cell below.

In [13]:
def find_agent(index: dict, skill_id: str) -> str:
    if skill_id not in index:
        return f"no agent offers {skill_id}"
    entry = index[skill_id]
    token = "needs a token" if entry["needs_token"] else "open"
    return f"{entry['agent']} at {entry['url']} ({token})"


for skill in ["lookup_order", "issue_refund", "track_parcel"]:
    print(find_agent(INDEX, skill))

Brightside orders agent at http://127.0.0.1:8270 (open)
Brightside refunds agent at http://127.0.0.1:8271 (needs a token)
no agent offers track_parcel


## Stop the agents
The two agents run in their own processes. Stop them at the end, and check that the ports are free again.

In [14]:
for proc in agents:
    proc.terminate()
    proc.wait(timeout=5)
print("ports free again:", not port_open(8270) and not port_open(8271))

ports free again: True


## Key terms
- **A2A** — the open protocol for one agent calling another agent over HTTP; version 1.0 is current (lesson 27).
- **Agent card** — the JSON document an agent publishes: name, endpoint, skills, capabilities and security (lesson 27).
- **Well-known URL** — a fixed path on a host where a standard document lives; the card is at `/.well-known/agent-card.json` (lesson 27).
- **Skill** — one thing an agent offers, with an id, a description and examples (lesson 27).
- **Security scheme** — how an agent accepts sign-in, such as a bearer token; a skill names the scheme it needs (lesson 27).
- **Registry** — a list of agent card URLs that a caller keeps and reads (lesson 27).